# #9 Memorias MoI (`Qwen/Qwen3.5-0.8B`, 1000 personas) — Colab L4

Misma prueba que `moi_memory_kaggle.ipynb`, pero en una L4 y **escribiendo 16 memorias a la vez** (cada una con su propio
delta y su propio Adam; los tests en CPU confirman que da lo mismo que una por una). En la T4 de Kaggle tardaba 23 s por memoria
(~8 h en total); aquí deberían ser unos ~1–2 s.

* **Reanudable:** las memorias escritas se guardan en Drive (`wsi_runs/moi_memory_qwen35-08b/progress.pt`) después de cada
  tanda. Si Colab se cae, vuelve a correr todo y sigue donde iba.
* **Resultado final:** `MyDrive/wsi_runs/FINALES/moi_memory_qwen35-08b.json` (solo si terminó). Luego el runtime se apaga solo.


In [ ]:
import os, json, shutil, subprocess, datetime
MODEL = "Qwen/Qwen3.5-0.8B"
RUN_NAME = "moi_memory_qwen35-08b"
REPO = "/content/wsi"
BASE = "/content/qwen-base"
from google.colab import drive
drive.mount("/content/drive")
DRIVE = f"/content/drive/MyDrive/wsi_runs/{RUN_NAME}"
FINALES = "/content/drive/MyDrive/wsi_runs/FINALES"
os.makedirs(DRIVE, exist_ok=True); os.makedirs(FINALES, exist_ok=True)
print("ya en Drive:", sorted(os.listdir(DRIVE)) or "nada")


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!pip install -q -U transformers
!pip install -q flash-linear-attention || echo "fla opcional"
import transformers, torch; print("transformers", transformers.__version__, "| GPU", torch.cuda.get_device_name(0))


In [ ]:
if not os.path.exists(REPO):
    !git clone -q --depth 1 https://github.com/krexito/weight-space-index.git {REPO}
else:
    !cd {REPO} && git pull -q
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
from huggingface_hub import snapshot_download
snapshot_download(MODEL, local_dir=BASE)
print("commit", COMMIT)


In [ ]:
# barrido de capa (12 personas aparte) + 1000 memorias (16 a la vez) + evaluación en 10/100/1000
OUT = f"{DRIVE}/results_moi_memory.json"
BANK = f"{DRIVE}/qwen35_0.8b_people_1000.moimem"
!python -u {REPO}/scripts/moi_memory.py --model {BASE} --dtype fp32 --tf32 --write-batch 16 --progress {DRIVE}/progress.pt --people 10,100,1000 --eval-people 50 --layers 3,5,7,9,11 --cov-tokens 100000 --ppl-tokens 32768 --tok-budget 8192 --save-bank {BANK} --out {OUT} 2>&1 | grep --line-buffered -v -E "Loading weights|fast path|Warning"


In [ ]:
# resultado final: a FINALES solo si la corrida terminó
res = json.load(open(OUT))
assert res.get("complete"), "la corrida no terminó: vuelve a correr la celda de arriba (sigue donde iba)"
res["_final"] = {"complete": True, "commit": COMMIT, "notebook": "moi_memory_colab.ipynb",
                 "gpu": torch.cuda.get_device_name(0),
                 "saved_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")}
dst = f"{FINALES}/{RUN_NAME}.json"
with open(dst + ".tmp", "w") as f:
    json.dump(res, f, indent=1, ensure_ascii=False)
os.replace(dst + ".tmp", dst)
print("Guardado final en Drive:", dst)
for N, sc in res["scales"].items():
    for arm, a in sc["arms"].items():
        p = a["probes"]
        print(N, arm.ljust(10), {k: p[k]["choice"] for k in ("write", "para", "es", "chat", "hop", "reverse", "nb_first", "nb_last") if k in p},
              "ppl", a.get("ppl", {}).get("ppl"), "caps", a.get("capitals_retained"))


In [ ]:
# Apagar el runtime: solo si el resultado final ya está en Drive
AUTO_APAGAR = True
import time
dst = f"{FINALES}/{RUN_NAME}.json"
ok = os.path.exists(dst) and json.load(open(dst)).get("_final", {}).get("complete")
if AUTO_APAGAR and ok:
    time.sleep(90)
    from google.colab import runtime
    runtime.unassign()
else:
    print("No apago: el resultado no está en FINALES todavía.")
